In [17]:
import librosa
import soundfile as sf
import numpy as np
from pydub import AudioSegment
from pydub.silence import split_on_silence
import os

def convert_and_preprocess(input_path, output_dir):
    """Convert .m4a to .wav and apply basic preprocessing"""
    
    # Step 1: Convert M4A → WAV (16kHz, mono — standard for emotion models)
    audio = AudioSegment.from_file(input_path, format="m4a")
    audio = audio.set_channels(1)           # Mono
    audio = audio.set_frame_rate(16000)     # 16kHz sample rate
    audio = audio.set_sample_width(2)       # 16-bit PCM
    
    wav_path = os.path.join(output_dir, "meeting_processed.wav")
    audio.export(wav_path, format="wav")
    
    return wav_path

In [18]:
import noisereduce as nr   # pip install noisereduce

def reduce_noise(wav_path, output_dir):
    y, sr = librosa.load(wav_path, sr=16000)
    
    # Use first 0.5s as noise profile (typically silence/background)
    noise_sample = y[:sr // 2]
    
    y_denoised = nr.reduce_noise(y=y, sr=sr, y_noise=noise_sample,
                                  prop_decrease=0.75)
    
    out_path = os.path.join(output_dir, "denoised.wav")
    sf.write(out_path, y_denoised, sr)
    return out_path

In [19]:
import webrtcvad
import wave
import struct

def apply_vad(wav_path, output_dir, aggressiveness=2):
    """
    aggressiveness: 0 (least aggressive) to 3 (most aggressive)
    Level 2 works well for meeting audio with background noise
    """
    vad = webrtcvad.Vad(aggressiveness)
    
    with wave.open(wav_path, 'rb') as wf:
        sample_rate = wf.getframerate()
        pcm_data = wf.readframes(wf.getnframes())
    
    # Process in 30ms frames (required by webrtcvad)
    frame_duration = 30  # ms
    frame_size = int(sample_rate * frame_duration / 1000) * 2
    
    voiced_frames = []
    for i in range(0, len(pcm_data) - frame_size, frame_size):
        frame = pcm_data[i:i + frame_size]
        if len(frame) == frame_size:
            is_speech = vad.is_speech(frame, sample_rate)
            if is_speech:
                voiced_frames.append(frame)
    
    out_path = os.path.join(output_dir, "vad_filtered.wav")
    with wave.open(out_path, 'wb') as out_wf:
        out_wf.setnchannels(1)
        out_wf.setsampwidth(2)
        out_wf.setframerate(sample_rate)
        out_wf.writeframes(b''.join(voiced_frames))
    
    return out_path

In [20]:
# pip install pyannote.audio  (requires HuggingFace token)
from pyannote.audio import Pipeline

def diarize_speakers(wav_path, hf_token, num_speakers=None):
    """Identify speaker turns — essential for per-speaker emotion analysis"""
    
    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        use_auth_token=hf_token
    )
    
    params = {"num_speakers": num_speakers} if num_speakers else {}
    diarization = pipeline(wav_path, **params)
    
    segments = []
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        segments.append({
            "speaker": speaker,
            "start": round(turn.start, 3),
            "end": round(turn.end, 3),
            "duration": round(turn.end - turn.start, 3)
        })
    
    return segments  # List of {speaker, start, end} dicts

In [21]:
def chunk_audio_by_segments(wav_path, segments, output_dir, min_duration=1.5):
    """
    Slice audio into per-speaker utterance chunks ready for annotation.
    Skips segments shorter than min_duration seconds.
    """
    y, sr = librosa.load(wav_path, sr=16000)
    chunks_meta = []
    
    os.makedirs(output_dir, exist_ok=True)
    
    for i, seg in enumerate(segments):
        duration = seg["end"] - seg["start"]
        if duration < min_duration:
            continue  # Skip very short turns
        
        start_sample = int(seg["start"] * sr)
        end_sample = int(seg["end"] * sr)
        chunk = y[start_sample:end_sample]
        
        filename = f"chunk_{i:04d}_{seg['speaker']}_{seg['start']:.1f}-{seg['end']:.1f}.wav"
        chunk_path = os.path.join(output_dir, filename)
        sf.write(chunk_path, chunk, sr)
        
        chunks_meta.append({
            **seg,
            "chunk_file": filename,
            "emotion_label": None,   # To be filled by annotator
            "intensity": None,       # e.g., low / medium / high
            "valence": None          # positive / negative / neutral
        })
    
    return chunks_meta

In [ ]:
# Main execution cell
import os
from pathlib import Path

# Configuration
input_audio = "C:/Users/VICTUS/Desktop/AudioStream/audio.m4a" # Change this to your audio file
output_base_dir = "./output"

# Create output directories
os.makedirs(output_base_dir, exist_ok=True)
convert_dir = os.path.join(output_base_dir, "01_converted")
denoise_dir = os.path.join(output_base_dir, "02_denoised")
vad_dir = os.path.join(output_base_dir, "03_vad")
chunks_dir = os.path.join(output_base_dir, "04_chunks")

os.makedirs(convert_dir, exist_ok=True)
os.makedirs(denoise_dir, exist_ok=True)
os.makedirs(vad_dir, exist_ok=True)
os.makedirs(chunks_dir, exist_ok=True)

# Pipeline execution
print("Step 1: Converting M4A to WAV...")
wav_file = convert_and_preprocess(input_audio, convert_dir)

print("Step 2: Reducing noise...")
denoised_file = reduce_noise(wav_file, denoise_dir)

print("Step 3: Applying voice activity detection...")
vad_file = apply_vad(denoised_file, vad_dir)

print("Step 4: Diarizing speakers...")
segments = diarize_speakers(vad_file, hf_token)
print(f"Found {len(segments)} speaker segments")

print("Step 5: Chunking audio by speaker...")
chunks_metadata = chunk_audio_by_segments(vad_file, segments, chunks_dir)
print(f"Created {len(chunks_metadata)} audio chunks")

# Display results
print("\nChunks created:")
for chunk in chunks_metadata[:5]:  # Show first 5
    print(f"  {chunk['chunk_file']} - Speaker: {chunk['speaker']}, Duration: {chunk['duration']}s")

Step 1: Converting M4A to WAV...
Step 2: Reducing noise...
Step 3: Applying voice activity detection...
Step 4: Diarizing speakers...


TypeError: Pipeline.from_pretrained() got an unexpected keyword argument 'use_auth_token'